# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a step-by-step guide for loading and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.### Dataset Source* Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport pprint# Define the Croissant metadata URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset metadata from the URLdataset = mlc.Dataset(croissant_url)# Print the dataset name and descriptionprint(f"Dataset: {dataset.metadata.name}\n{dataset.metadata.description}")

## 2. Data Overview
List available **record sets** and their fields. All entities are referenced by their `@id` as specified in the Croissant schema.
We'll print the `@id`, `name`, and main fields for each record set found in this dataset.

In [ ]:
# Get record sets and their information
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")for rs in record_sets:
    print(f"@id: {rs.id}")
    print(f"  name: {rs.name}")    if hasattr(rs, 'description') and rs.description:
        print(f"  description: {rs.description}")    print(f"  Fields (@id):")    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name})")    print('-' * 60)

## 3. Data Extraction
Let's load data from the main record set into a DataFrame for analysis.

---
Below, we first select the main record set (with its `@id`) and load it. You can adapt this to other record sets as needed.

In [ ]:
# Typically there is one primary record set. Let's use the first one found.main_rs = record_sets[0]main_rs_id = main_rs.id# List all record sets IDsrecord_set_ids = [rs.id for rs in record_sets]print("Record Set @id list:", record_set_ids)# Load all records from each record set into a DataFramedataframes = {}for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    dataframes[rs.id] = pd.DataFrame(records)print(f"Columns in `{main_rs_id}` record set:")print(dataframes[main_rs_id].columns.tolist())# Show a previewdataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's apply some data processing steps on this record set's DataFrame.
You can select a numeric field of interest (referenced by its `@id` – see previous section output), and perform operations like filtering, normalization, and grouping by a categorical field.

In [ ]:
# -- Pick numeric and group fields by their @id --# Example field IDs obtained from section above. Update if desired.numeric_field_id = None
for field in main_rs.fields:
    # Heuristically pick the first numeric field (Float or Integer)
    if hasattr(field, 'data_type') and field.data_type in ('Float', 'Integer', 'Number'):
        numeric_field_id = field.id
        break
print(f"Numeric field selected: {numeric_field_id}")df = dataframes[main_rs_id]# Ensure the column is numericif numeric_field_id is not None:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')threshold = df[numeric_field_id].mean() if numeric_field_id else 0filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id else dfprint(f"Filtered rows where {numeric_field_id} > mean:")print(filtered_df.head())if numeric_field_id:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())# Try grouping by the first non-numeric fieldgroup_field_id = Nonefor field in main_rs.fields:
    if field.id != numeric_field_id and hasattr(field, 'data_type') and field.data_type not in ('Float', 'Integer', 'Number'):
        group_field_id = field.id
        break
print(f"Group field selected: {group_field_id}")if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version.
All charts use only fields referenced by their `@id`s.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns# Histogram of numeric fieldif numeric_field_id:
    plt.figure(figsize=(10, 5))    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)    plt.title(f'{numeric_field_id} Distribution')    plt.xlabel(numeric_field_id)    plt.ylabel('Count')    plt.show()# If a normalized column exists, plot that toonorm_col = f"{numeric_field_id}_normalized"if norm_col in filtered_df.columns:
    plt.figure(figsize=(10, 5))    sns.histplot(filtered_df[norm_col].dropna(), bins=30, kde=True, color='orange')    plt.title(f'Normalized {numeric_field_id} Distribution (filtered)')    plt.xlabel(norm_col)    plt.ylabel('Count')    plt.show()# Bar plot for grouped mean if grouping is availableif 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(12, 5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
* Using the Croissant metadata, we programmatically explored the structure of the FAIR^2 dataset.
* Data was extracted using only the unique `@id` for each record set and field, ensuring reproducible, unambiguous analytics.
* Simple filtering, normalization, grouping, and visualization demonstrates how to use `mlcroissant` for rapid Fair Data onboarding and exploration.

You can further adapt this notebook by specifying additional field `@id`s, more advanced feature engineering, or by exploring additional record sets as your analysis requires.